In [ ]:
#to prevent colab to automatically disconnect
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button")
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click()
     }

   btn = document.getElementById('ok')
   if (btn != null){
     console.log("Click reconnect");
     btn.click()
     }
  }

setInterval(ClickConnect,60000)
'''))

print("Done.")

<IPython.core.display.Javascript object>

Done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install xgboost

# **Step 1 : Importing Libraries**

In [ ]:
# For data processing
import numpy as np
import math
from math import sqrt

# For data processing and manipulation
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# For checking path
import os
import json
import joblib

#metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

#tensorflow libs
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping , Callback
import tensorflow as tf
from tensorflow.keras import backend as K
from keras import backend

from tensorflow.keras.layers import *
from tensorflow.keras.layers import Dense , GRU ,Dropout , PReLU , RepeatVector ,TimeDistributed, Attention,LayerNormalization,Add,Activation
from tensorflow.keras.models import Sequential,load_model,Model
from tensorflow.keras.utils import to_categorical , plot_model
from tensorflow.keras import regularizers, constraints, initializers, activations

from sklearn.linear_model import LinearRegression
import xgboost as xgb
from lightgbm import LGBMRegressor

from tensorflow.keras.layers import InputSpec

#from keras_self_attention import SeqSelfAttention
from tensorflow.keras.layers import Concatenate

from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Bidirectional, TimeDistributed, LayerNormalization, Add
from tensorflow.keras import layers


tf.get_logger().setLevel('ERROR')
mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

# **Step 2 : Loading Dataset and Models**

In [ ]:
path_save_models = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models'

In [ ]:
patrol_divisons ={1: 'Central',
 2: 'Rampart',
 3: 'Southwest',
 4: 'Hollenbeck',
 5: 'Harbor',
 6: 'Hollywood',
 7: 'Wilshire',
 8: 'West LA',
 9: 'Van Nuys',
 10: 'West Valley',
 11: 'Northeast',
 12: '77th Street',
 13: 'Newton',
 14: 'Pacific',
 15: 'N Hollywood',
 16: 'Foothill',
 17: 'Devonshire',
 18: 'Southeast',
 19: 'Mission',
 20: 'Olympic',
 21: 'Topanga'}

In [ ]:
#Loading BI GRU Models
model_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention Bi GRU/mh_attn_bi_gru_model_files'
bi_gru_models = {}
for divison in patrol_divisons.keys():
  path = os.path.join(model_path,f"mh_attn_bi_gru_all_feature_{divison}.keras")
  bi_gru_models[divison] = load_model(path)


In [ ]:
#Loading CNN BI GRU Models
model_path2 =r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention CNN Bi GRU/mh_attn_cnn_bi_gru_model_files'
cnn_bi_gru_models = {}
for divison in patrol_divisons.keys():
  path = os.path.join(model_path2,f"mh_attn_cnn_bi_gru_{divison}.keras")
  cnn_bi_gru_models[divison] = load_model(path)

In [ ]:
dataset_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset'

In [ ]:
files = os.listdir(dataset_path)
files

['Central_1.csv',
 'Rampart_2.csv',
 'Southwest_3.csv',
 'Hollenbeck_4.csv',
 'Harbor_5.csv',
 'Hollywood_6.csv',
 'Wilshire_7.csv',
 'West LA_8.csv',
 'Van Nuys_9.csv',
 'West Valley_10.csv',
 'Northeast_11.csv',
 '77th Street_12.csv',
 'Newton_13.csv',
 'Pacific_14.csv',
 'N Hollywood_15.csv',
 'Foothill_16.csv',
 'Devonshire_17.csv',
 'Southeast_18.csv',
 'Mission_19.csv',
 'Olympic_20.csv',
 'Topanga_21.csv']

In [ ]:
#loading dataset
patrol_divisons = {}
dataset = {}
for file in files:
  name = file.split('_')[0]
  id = file.split('_')[1].split('.')[0]
  patrol_divisons[int(id)] = name
  dataset[int(id)] = pd.read_csv(os.path.join(dataset_path,file))
  if 'Unnamed: 0' in dataset[int(id)].columns:
        dataset[int(id)] = dataset[int(id)].drop('Unnamed: 0', axis=1)


patrol_divisons = dict(sorted(patrol_divisons.items()))
patrol_divisons

{1: 'Central',
 2: 'Rampart',
 3: 'Southwest',
 4: 'Hollenbeck',
 5: 'Harbor',
 6: 'Hollywood',
 7: 'Wilshire',
 8: 'West LA',
 9: 'Van Nuys',
 10: 'West Valley',
 11: 'Northeast',
 12: '77th Street',
 13: 'Newton',
 14: 'Pacific',
 15: 'N Hollywood',
 16: 'Foothill',
 17: 'Devonshire',
 18: 'Southeast',
 19: 'Mission',
 20: 'Olympic',
 21: 'Topanga'}

In [ ]:
columns_in_dataset = dataset[1].columns
columns_in_dataset

Index(['datetime', 'p_id', '1', '2', '3', '4', '5', '6', '7', '8', 'group 0',
       'count', 'day sin', 'day cos', 'week sin', 'week cos', 'year sin',
       'year cos'],
      dtype='object')

In [ ]:
dataset[1].head()

,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,2010-01-01 00:00:00,1,6.0,1.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0,11.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,2010-01-01 03:00:00,1,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2010-01-01 06:00:00,1,0.0,0.0,0.0,12.0,1.0,0.0,0.0,1.0,0.0,14.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,2010-01-01 09:00:00,1,2.0,0.0,0.0,13.0,1.0,0.0,1.0,0.0,0.0,17.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,2010-01-01 12:00:00,1,0.0,2.0,0.0,7.0,0.0,0.0,0.0,1.0,0.0,10.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


# **Step 3: Converting to TimeSeries data**

Utility Function

In [ ]:
def train_test_val_split(dataset):
  columns_indices = {name:i for i,name in enumerate(dataset.columns)}
  n = len(dataset)

  #splitting dataset
  training_set = dataset[:int(n*0.7)]
  validation_set = dataset[int(n*0.7):int(n*0.9)]
  test_set = dataset[int(n*0.9):]

  num_features = dataset.shape[1]
  return training_set, validation_set, test_set, num_features, columns_indices

In [ ]:
class WindowGenerator():

    def __init__(self, input_width, label_width, shift,
               train_df, val_df, test_df,
               label_columns=None , shuffle=False , batch_size = 64):
        '''
        The __init__ method includes all the necessary logic for the input and label indices.
        Input:
            input_width : input width / window size
            label_width : output width
            shift : size of window shifting forward
            train_df : train dataset
            val_df : validation dataset
            test_df : test dataset
            label_columns ( Default = None) : Label Columns
            shuffle ( Default = False) : weather to shuffle data
            batch_size (Default = 64) : Batch Size
        Output: None
        Example :
            w2 = WindowGenerator(input_width=6, label_width=1, shift=1,
                     label_columns=['count'])
            w2

        '''
        # Store the raw data.
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self.shuffle = shuffle
        self.batch_size = batch_size

        # Work out the label column indices.
        self.label_columns = label_columns
        if label_columns is not None:
            self.label_columns_indices = {name: i for i, name in
                                        enumerate(label_columns)}

        self.column_indices = {name: i for i, name in
                            enumerate(train_df.columns)}


        # Work out the window parameters.
        self.input_width = input_width
        self.label_width = label_width
        self.shift = shift

        self.total_window_size = input_width + shift

        self.input_slice = slice(0, input_width) #(start , stop)
        self.input_indices = np.arange(self.total_window_size)[self.input_slice]

        self.label_start = self.total_window_size - self.label_width
        self.labels_slice = slice(self.label_start, None)
        self.label_indices = np.arange(self.total_window_size)[self.labels_slice]

    def __repr__(self):
        return '\n'.join([
            f'Total window size: {self.total_window_size}',
            f'Input indices: {self.input_indices}',
            f'Label indices: {self.label_indices}',
            f'Label column name(s): {self.label_columns}'])

    def split_window(self, features):

        inputs = features[:, self.input_slice, :]
        labels = features[:, self.labels_slice, :]
        #taking only the labels that are presentin the label_columns
        if self.label_columns is not None:
            labels = tf.stack([labels[:, :, self.column_indices[name]] for name in self.label_columns],axis=-1)

        # Slicing doesn't preserve static shape information, so set the shapes
        # manually. This way the `tf.data.Datasets` are easier to inspect.
        inputs.set_shape([None, self.input_width, None])
        labels.set_shape([None, self.label_width, None])

        return inputs, labels




    def make_dataset(self, data):

        data = np.array(data, dtype=np.float32)
        ds = tf.keras.preprocessing.timeseries_dataset_from_array(
            data=data,
            targets=None,
            sequence_length=self.total_window_size,
            sequence_stride=1,
            shuffle=self.shuffle,
            batch_size=self.batch_size,)
        ds = ds.map(self.split_window)
        return ds

    def create_dataset2(self , map_df , reshape=True):
      x = []
      y = []
      for res in iter(map_df):
        inputs, labels = res
        if(len(inputs)==64):
          x.append(inputs)
          y.append(labels)

      x = np.array(x)
      y = np.array(y)
      if(reshape):
        x = x.reshape(-1, x.shape[-2] , x.shape[-1])
        y = y.reshape(-1 , y.shape[-2] , y.shape[-1])
      return x , y


    @property
    def train(self):
        return self.make_dataset(self.train_df)

    @property
    def val(self):
        return self.make_dataset(self.val_df)

    @property
    def test(self):
        return self.make_dataset(self.test_df)



In [ ]:
def create_data(train , test , val , columns):
    '''
    Create dataset from main train , test , val with given columns
    '''
    if(columns==None):
        columns = train.columns
    new_train = train[columns]
    new_test = test[columns]
    new_val = val[columns]
    return new_train , new_test , new_val

In [ ]:
def save_history(history , path):
    # convert the history.history dict to a pandas DataFrame:
    hist_df = pd.DataFrame(history.history)
    # or save to csv:
    hist_csv_file = path
    with open(hist_csv_file, mode='w') as f:
        hist_df.to_json(f)

def get_history(path):
	with open(path) as json_file:
		data = json.load(json_file)
		return data

def save_model_weights(model , path):
  model.save_weights(path)


Function to save results

In [ ]:
def build_metrics_dataframe(test_metrics_dict, val_metrics_dict, patrol_divisions_dict):
    """
    Converts performance dictionaries into a structured DataFrame.

    Parameters:
        test_metrics_dict (dict): Dict of test evaluation results {division_id: [loss, rmse, mae]}
        val_metrics_dict (dict): Dict of val evaluation results {division_id: [loss, rmse, mae]}
        patrol_divisions_dict (dict): Dict mapping division_id to division name

    Returns:
        pd.DataFrame: DataFrame with division names as rows and metrics as columns
    """
    rows = []

    for div_id in test_metrics_dict.keys():
        div_name = patrol_divisions_dict[int(div_id)]
        test = test_metrics_dict[div_id]
        val = val_metrics_dict[div_id]

        rows.append({
            "Division": div_name,
            "test rmse": test[1],
            "test mae": test[2],
            "val rmse": val[1],
            "val mae": val[2]
        })

    df = pd.DataFrame(rows)
    df.set_index("Division", inplace=True)
    return df


In [ ]:
#without temperature
general_indexs = ['1', '2', '3', '4', '5', '6', '7','8', 'count',
           'day sin', 'day cos', 'week sin', 'week cos',
           'year sin', 'year cos', 'group 0']

x_col = ['day sin' , 'day cos' , 'year sin' , 'year cos' , 'week cos' , 'week sin' ,'datetime']

def generate_window(df_now, ret_test = 0):
    train_df , val_df , test_df , num_features_df , column_indices_df = train_test_val_split(df_now)
    train_df , test_df , val_df = create_data(train_df , test_df , val_df , general_indexs)
    y_col = []

    #storing label columns
    for i in train_df.columns:
        if (i in x_col):
            continue
        y_col.append(i)

    #creating window generator object
    wide_window_all = WindowGenerator(train_df=train_df, test_df=test_df , val_df=val_df,
        input_width=24, label_width=24, shift=1,
        label_columns=y_col)

    if (ret_test == 1):
      return wide_window_all, test_df
    else:
      return wide_window_all


# **Step 4: Decision Level Fusion**

Preparing Dataset

In [ ]:
# Dictionary to store data per division
division_data_dict = {}

# Loop over each division in your dataset
for division in patrol_divisons.keys():

    # Generate windowed validation and test datasets
    wide_window_val = generate_window(dataset[division])
    val_data = wide_window_val.val

    wide_window_test = generate_window(dataset[division])
    test_data = wide_window_test.test

    # Unbatch and convert validation data to numpy arrays
    x_val_list, y_val_list = [], []
    for x, y in val_data.unbatch():
        x_val_list.append(x.numpy())
        y_val_list.append(y.numpy())
    x_val = np.array(x_val_list)
    y_val = np.array(y_val_list)

    # Unbatch and convert test data to numpy arrays
    x_test_list, y_test_list = [], []
    for x, y in test_data.unbatch():
        x_test_list.append(x.numpy())
        y_test_list.append(y.numpy())
    x_test = np.array(x_test_list)
    y_test = np.array(y_test_list)

    # Store in dictionary
    division_data_dict[division] = {
        'x_val': x_val,
        'y_val': y_val,
        'x_test': x_test,
        'y_test': y_test
    }

In [ ]:
x_val_dataset_shape = {}
x_test_dataset_shape = {}
y_val_dataset_shape = {}
y_test_dataset_shape = {}
for i in division_data_dict.keys():
  x_val_dataset_shape[i] = division_data_dict[i]['x_val'].shape
  x_test_dataset_shape[i] = division_data_dict[i]['x_test'].shape
  y_val_dataset_shape[i] = division_data_dict[i]['y_val'].shape
  y_test_dataset_shape[i] = division_data_dict[i]['y_test'].shape

In [ ]:
for i in x_val_dataset_shape.keys():
  print(f"Division {i}")
  print("x_val shape:", x_val_dataset_shape[i])
  print("y_val shape:", y_val_dataset_shape[i])
  print("x_test shape:", x_test_dataset_shape[i])
  print("y_test shape:", y_test_dataset_shape[i])
  print()


Division 1
x_val shape: (8811, 24, 16)
y_val shape: (8811, 24, 10)
x_test shape: (4393, 24, 16)
y_test shape: (4393, 24, 10)

Division 2
x_val shape: (8785, 24, 16)
y_val shape: (8785, 24, 10)
x_test shape: (4381, 24, 16)
y_test shape: (4381, 24, 10)

Division 3
x_val shape: (8855, 24, 16)
y_val shape: (8855, 24, 10)
x_test shape: (4416, 24, 16)
y_test shape: (4416, 24, 10)

Division 4
x_val shape: (8856, 24, 16)
y_val shape: (8856, 24, 10)
x_test shape: (4416, 24, 16)
y_test shape: (4416, 24, 10)

Division 5
x_val shape: (8857, 24, 16)
y_val shape: (8857, 24, 10)
x_test shape: (4417, 24, 16)
y_test shape: (4417, 24, 10)

Division 6
x_val shape: (8739, 24, 16)
y_val shape: (8739, 24, 10)
x_test shape: (4358, 24, 16)
y_test shape: (4358, 24, 10)

Division 7
x_val shape: (8740, 24, 16)
y_val shape: (8740, 24, 10)
x_test shape: (4359, 24, 16)
y_test shape: (4359, 24, 10)

Division 8
x_val shape: (8832, 24, 16)
y_val shape: (8832, 24, 10)
x_test shape: (4405, 24, 16)
y_test shape: (4405, 2

In [ ]:
# Get predicted y values
bi_gru_pred_for_val = {}
cnn_bi_gru_pred_for_val = {}
bi_gru_pred_for_val_shape ={}
cnn_bi_gru_pred_for_val_shape ={}

for i in division_data_dict.keys():

    #predicting for  validation set using bi-gru
    bi_gru_pred_for_val[i] = bi_gru_models[i].predict(division_data_dict[i]['x_val'])
    bi_gru_pred_for_val_shape[i] = bi_gru_pred_for_val[i].shape

    #predicting for validation set using cnn-bi-gru
    cnn_bi_gru_pred_for_val[i] = cnn_bi_gru_models[i].predict(division_data_dict[i]['x_val'])
    cnn_bi_gru_pred_for_val_shape[i] = cnn_bi_gru_pred_for_val[i].shape

276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step
275/275 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step
275/275 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
277/277 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step
277/277 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
277/277 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step
277/277 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
277/277 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step
277/277 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
274/274 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step
274/274 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
274/274 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step
274/274 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step
277/277 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step
277/277 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step
276/

In [ ]:
# Get predicted y values
bi_gru_pred_for_test = {}
cnn_bi_gru_pred_for_test = {}
bi_gru_pred_for_test_shape ={}
cnn_bi_gru_pred_for_test_shape ={}

for i in division_data_dict.keys():

    #predicting for  test set using bi-gru
    bi_gru_pred_for_test[i] = bi_gru_models[i].predict(division_data_dict[i]['x_test'])
    bi_gru_pred_for_test_shape[i] = bi_gru_pred_for_test[i].shape

    #predicting for test set using cnn-bi-gru
    cnn_bi_gru_pred_for_test[i] = cnn_bi_gru_models[i].predict(division_data_dict[i]['x_test'])
    cnn_bi_gru_pred_for_test_shape[i] = cnn_bi_gru_pred_for_val[i].shape

138/138 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
139/139 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step
138/138 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
138/13

# **1.SIMPLE AVERAGE**

In [ ]:
# Dictionary to store RMSE and MAE for each division
simple_average_for_test = {}

# Loop over each division
for division_id in patrol_divisons.keys():
    # Assume you have predictions and true values already computed per division
    y_pred = (bi_gru_pred_for_test[division_id] + cnn_bi_gru_pred_for_test[division_id]) / 2

    y_test = division_data_dict[division_id]['y_test']

    # Flatten the arrays
    y_test_flat = y_test.reshape(-1)
    y_pred_flat = y_pred.reshape(-1)

    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test_flat, y_pred_flat))
    mae = mean_absolute_error(y_test_flat, y_pred_flat)

    # Store in dictionary
    simple_average_for_test[division_id] = {
        'test_rmse': rmse,
        'test_mae': mae
    }

In [ ]:
simple_average_for_test

{1: {'test_rmse': np.float64(0.7049670145082843),
  'test_mae': 0.18260590732097626},
 2: {'test_rmse': np.float64(0.40083623140890745),
  'test_mae': 0.09010086953639984},
 3: {'test_rmse': np.float64(0.34908598609946007),
  'test_mae': 0.09033490717411041},
 4: {'test_rmse': np.float64(0.1801879795979744),
  'test_mae': 0.053512658923864365},
 5: {'test_rmse': np.float64(0.19504361717854504),
  'test_mae': 0.052165694534778595},
 6: {'test_rmse': np.float64(0.25098306372864654),
  'test_mae': 0.06670486927032471},
 7: {'test_rmse': np.float64(0.24566545000501813),
  'test_mae': 0.07487944513559341},
 8: {'test_rmse': np.float64(0.20350888546787446),
  'test_mae': 0.04861847311258316},
 9: {'test_rmse': np.float64(0.19803013288770663),
  'test_mae': 0.050175219774246216},
 10: {'test_rmse': np.float64(0.19965152855265753),
  'test_mae': 0.04927586019039154},
 11: {'test_rmse': np.float64(0.20421849784164214),
  'test_mae': 0.05165375769138336},
 12: {'test_rmse': np.float64(0.26568231

In [ ]:
# Convert the nested dictionary to a DataFrame
simple_avg_df = pd.DataFrame.from_dict(simple_average_for_test, orient='index')

# Reset index and rename the column to something meaningful
simple_avg_df.index.name = 'Patrol Division'
simple_avg_df.reset_index(inplace=True)

# View the DataFrame
print(simple_avg_df)


    Patrol Division  test_rmse  test_mae
0                 1   0.704967  0.182606
1                 2   0.400836  0.090101
2                 3   0.349086  0.090335
3                 4   0.180188  0.053513
4                 5   0.195044  0.052166
5                 6   0.250983  0.066705
6                 7   0.245665  0.074879
7                 8   0.203509  0.048618
8                 9   0.198030  0.050175
9                10   0.199652  0.049276
10               11   0.204218  0.051654
11               12   0.265682  0.087364
12               13   0.296724  0.079319
13               14   0.272848  0.084493
14               15   0.367511  0.097617
15               16   0.182634  0.058782
16               17   0.211627  0.065880
17               18   0.242090  0.076902
18               19   0.204350  0.069834
19               20   0.298217  0.077862
20               21   0.205995  0.067980


In [ ]:
path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/Results/DLF RESULTS'
save_path = os.path.join(path,"simple_average_dlf_results.csv")
simple_avg_df.to_csv(save_path, index=False)

# **2.Weighted Average**

In [ ]:
# Store best weights and metrics
best_weight_results_bi_gru_dominant = {}

# Weight list (BiGRU-heavy)
weight_options = [0.6, 0.7, 0.8, 0.9, 1.0]

for division_id in patrol_divisons.keys():
    y_test = division_data_dict[division_id]['y_test'].reshape(-1)

    best_rmse = float('inf')
    best_weight = None
    best_mae = None

    for alpha in weight_options:
        # Weighted prediction
        y_pred = alpha * bi_gru_pred_for_test[division_id] + (1 - alpha) * cnn_bi_gru_pred_for_test[division_id]
        y_pred_flat = y_pred.reshape(-1)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred_flat))
        mae = mean_absolute_error(y_test, y_pred_flat)

        # Track best
        if rmse < best_rmse:
            best_rmse = rmse
            best_mae = mae
            best_weight = alpha

    best_weight_results_bi_gru_dominant[division_id] = {
        'best_bi_gru_weight': best_weight,
        'corresponding_cnn_bi_gru_weight': round(1 - best_weight, 2),
        'best_rmse': best_rmse,
        'best_mae': best_mae
    }

# Convert to DataFrame
best_weight_bi_gru_df = pd.DataFrame.from_dict(best_weight_results_bi_gru_dominant, orient='index')
best_weight_bi_gru_df.index.name = 'Patrol Division'
best_weight_bi_gru_df.reset_index(inplace=True)

In [ ]:
# Show output
best_weight_bi_gru_df.head(21)


,Patrol Division,best_bi_gru_weight,corresponding_cnn_bi_gru_weight,best_rmse,best_mae
0,1,0.6,0.4,0.703575,0.184667
1,2,0.7,0.3,0.399779,0.091154
2,3,0.6,0.4,0.347180,0.087599
3,4,0.6,0.4,0.180216,0.054463
4,5,0.8,0.2,0.193526,0.052026
5,6,0.8,0.2,0.248895,0.063859
6,7,1.0,0.0,0.227689,0.051474
7,8,0.8,0.2,0.201931,0.044869
8,9,0.8,0.2,0.196184,0.045731
9,10,0.7,0.3,0.198842,0.046799


In [ ]:
#save to CSV
saving_path1 = os.path.join(path,"weighted_average.csv")
best_weight_bi_gru_df.to_csv(saving_path1, index=False)

# **3.Stacking-Based Decision-Level Fusion (DLF)**

Dataset Preparation

In [ ]:
bi_gru_val_preds = {}
cnn_bi_gru_val_preds = {}
bi_gru_test_preds = {}
cnn_bi_gru_test_preds = {}
val_y = {}
test_y = {}
X_val_stack = {}
y_val_stack = {}
X_test_stack = {}

for i in patrol_divisons.keys():

  # Flattening Validation set
  bi_gru_val_preds[i] = bi_gru_pred_for_val[i].reshape(-1)
  cnn_bi_gru_val_preds[i] = cnn_bi_gru_pred_for_val[i].reshape(-1)

  #Flattening Test set
  bi_gru_test_preds[i] = bi_gru_pred_for_test[i].reshape(-1)
  cnn_bi_gru_test_preds[i] = cnn_bi_gru_pred_for_test[i].reshape(-1)


  val_y[i] = division_data_dict[i]["y_val"].reshape(-1)
  test_y[i] = division_data_dict[i]["y_test"].reshape(-1)

  X_val_stack[i] = np.column_stack((bi_gru_val_preds[i], cnn_bi_gru_val_preds[i]))
  y_val_stack[i] = val_y[i]  # true validation targets

  # Stack model predictions for test set (input for final prediction)
  X_test_stack[i] = np.column_stack((bi_gru_test_preds[i], cnn_bi_gru_test_preds[i]))

In [ ]:
lr_models_save_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lr_models'
xgb_models_save_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/xgb_models'
lgbm_models_save_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lgbm_models'

**3.1.Linear Regression**

In [ ]:
lr_model = {}
lr_preds = {}

In [ ]:
for i in patrol_divisons.keys():
  lr_model[i] = LinearRegression()
  lr_model[i].fit(X_val_stack[i], y_val_stack[i])
  lr_preds[i] = lr_model[i].predict(X_test_stack[i])
  model_path = os.path.join(lr_models_save_path, f"lr_model_{i}.pkl")
  joblib.dump(lr_model[i], model_path)

In [ ]:
lr= {}
for i in patrol_divisons.keys():

  rmse = np.sqrt(mean_squared_error(test_y[i], lr_preds[i]))
  mae = mean_absolute_error(test_y[i], lr_preds[i])
  lr[i] = {'Division':i,'test rmse ':rmse ,'test mae ':mae}
  print(f"Division {i} : LR Test RMSE: {rmse:.4f}, LR Test MAE: {mae:.4f}")

Division 1 : LR Test RMSE: 0.6886, LR Test MAE: 0.1922
Division 2 : LR Test RMSE: 0.3985, LR Test MAE: 0.0987
Division 3 : LR Test RMSE: 0.3496, LR Test MAE: 0.0989
Division 4 : LR Test RMSE: 0.1772, LR Test MAE: 0.0546
Division 5 : LR Test RMSE: 0.1934, LR Test MAE: 0.0580
Division 6 : LR Test RMSE: 0.2479, LR Test MAE: 0.0861
Division 7 : LR Test RMSE: 0.2230, LR Test MAE: 0.0695
Division 8 : LR Test RMSE: 0.2024, LR Test MAE: 0.0643
Division 9 : LR Test RMSE: 0.1963, LR Test MAE: 0.0588
Division 10 : LR Test RMSE: 0.1982, LR Test MAE: 0.0623
Division 11 : LR Test RMSE: 0.2024, LR Test MAE: 0.0621
Division 12 : LR Test RMSE: 0.2606, LR Test MAE: 0.0893
Division 13 : LR Test RMSE: 0.2888, LR Test MAE: 0.0773
Division 14 : LR Test RMSE: 0.2592, LR Test MAE: 0.0890
Division 15 : LR Test RMSE: 0.3550, LR Test MAE: 0.1016
Division 16 : LR Test RMSE: 0.1760, LR Test MAE: 0.0576
Division 17 : LR Test RMSE: 0.2013, LR Test MAE: 0.0640
Division 18 : LR Test RMSE: 0.2359, LR Test MAE: 0.0702
D

In [ ]:
lr_df = pd.DataFrame.from_dict(lr, orient='index')
lr_df.index.name = "Division"

In [ ]:
lr_df.head(21)

,Division,test rmse,test mae
Division,,,
1,1,0.688575,0.192168
2,2,0.398458,0.098691
3,3,0.349630,0.098854
4,4,0.177243,0.054586
5,5,0.193351,0.057960
6,6,0.247907,0.086125
7,7,0.222979,0.069478
8,8,0.202353,0.064272
9,9,0.196297,0.058760


In [ ]:
#save to CSV
saving_path2 = os.path.join(path,"linear_regression.csv")
lr_df.to_csv(saving_path2, index=False)

**3.2.XG Boost**

In [ ]:
xgb_meta = {}
xgb_preds = {}
xgb = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [ ]:
for i in patrol_divisons.keys():
    xgb_meta[i] = xgb.fit(X_val_stack[i], y_val_stack[i])
    xgb_preds[i] = xgb_meta[i].predict(X_test_stack[i])
    model_path = os.path.join(xgb_models_save_path, f"xgb_model_{i}.pkl")
    joblib.dump(xgb_meta[i], model_path)

In [ ]:
xgb_results= {}
for i in patrol_divisons.keys():

  rmse = np.sqrt(mean_squared_error(test_y[i], xgb_preds[i]))
  mae = mean_absolute_error(test_y[i], xgb_preds[i])
  xgb_results[i] = {'Division':i,'test rmse ':rmse ,'test mae ':mae}
  print(f"Division {i} : LR Test RMSE: {rmse:.4f}, LR Test MAE: {mae:.4f}")

Division 1 : LR Test RMSE: 0.6725, LR Test MAE: 0.1750
Division 2 : LR Test RMSE: 0.4044, LR Test MAE: 0.0816
Division 3 : LR Test RMSE: 0.3525, LR Test MAE: 0.0846
Division 4 : LR Test RMSE: 0.1835, LR Test MAE: 0.0397
Division 5 : LR Test RMSE: 0.2014, LR Test MAE: 0.0456
Division 6 : LR Test RMSE: 0.2485, LR Test MAE: 0.0745
Division 7 : LR Test RMSE: 0.2363, LR Test MAE: 0.0627
Division 8 : LR Test RMSE: 0.2083, LR Test MAE: 0.0464
Division 9 : LR Test RMSE: 0.2025, LR Test MAE: 0.0476
Division 10 : LR Test RMSE: 0.2045, LR Test MAE: 0.0502
Division 11 : LR Test RMSE: 0.2071, LR Test MAE: 0.0496
Division 12 : LR Test RMSE: 0.2714, LR Test MAE: 0.0780
Division 13 : LR Test RMSE: 0.2984, LR Test MAE: 0.0640
Division 14 : LR Test RMSE: 0.2658, LR Test MAE: 0.0667
Division 15 : LR Test RMSE: 0.3673, LR Test MAE: 0.0935
Division 16 : LR Test RMSE: 0.1765, LR Test MAE: 0.0448
Division 17 : LR Test RMSE: 0.2123, LR Test MAE: 0.0535
Division 18 : LR Test RMSE: 0.2485, LR Test MAE: 0.0616
D

In [ ]:
xgb_df = pd.DataFrame.from_dict(xgb_results, orient='index')
xgb_df.index.name = "Division"

In [ ]:
xgb_df.head(21)

,Division,test rmse,test mae
Division,,,
1,1,0.672450,0.174976
2,2,0.404401,0.081569
3,3,0.352477,0.084600
4,4,0.183516,0.039725
5,5,0.201440,0.045634
6,6,0.248468,0.074465
7,7,0.236334,0.062691
8,8,0.208317,0.046365
9,9,0.202499,0.047584


In [ ]:
#save to CSV
saving_path3 = os.path.join(path,"XG_Boost.csv")
xgb_df.to_csv(saving_path3, index=False)

**3.3 LightGBM Regressor**

In [ ]:
lgb_models = {}
lgb_preds = {}
lgb_meta = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

In [ ]:
for i in patrol_divisons.keys():
    lgb_models[i] = lgb_meta.fit(X_val_stack[i],  y_val_stack[i])
    lgb_preds[i] = lgb_models[i].predict(X_test_stack[i])
    model_path = os.path.join(lgbm_models_save_path, f"lgb_model_{i}.pkl")
    joblib.dump(lgb_models[i], model_path)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044302 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 2114640, number of used features: 2
[LightGBM] [Info] Start training from score 1.031564
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073843 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 2108400, number of used features: 2
[LightGBM] [Info] Start training from score 0.705013
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043078 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 2118480, number of used features: 2
[LightGBM] [Info] Start training from score 0.771435
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.040205 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 2125440, number of used features: 2
[LightGBM] [Info] Start training from score 0.611385
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
lgb_results= {}
for i in patrol_divisons.keys():
  rmse = np.sqrt(mean_squared_error(test_y[i], lgb_preds[i]))
  mae = mean_absolute_error(test_y[i], lgb_preds[i])
  lgb_results[i] = {'Division':i,'test rmse ':rmse ,'test mae ':mae}
  print(f"Division {i} : LR Test RMSE: {rmse:.4f}, LR Test MAE: {mae:.4f}")

Division 1 : LR Test RMSE: 0.6718, LR Test MAE: 0.1724
Division 2 : LR Test RMSE: 0.4044, LR Test MAE: 0.0813
Division 3 : LR Test RMSE: 0.3630, LR Test MAE: 0.0858
Division 4 : LR Test RMSE: 0.1837, LR Test MAE: 0.0400
Division 5 : LR Test RMSE: 0.1999, LR Test MAE: 0.0438
Division 6 : LR Test RMSE: 0.2477, LR Test MAE: 0.0709
Division 7 : LR Test RMSE: 0.2359, LR Test MAE: 0.0621
Division 8 : LR Test RMSE: 0.2083, LR Test MAE: 0.0482
Division 9 : LR Test RMSE: 0.2021, LR Test MAE: 0.0465
Division 10 : LR Test RMSE: 0.2042, LR Test MAE: 0.0484
Division 11 : LR Test RMSE: 0.2068, LR Test MAE: 0.0486
Division 12 : LR Test RMSE: 0.2656, LR Test MAE: 0.0723
Division 13 : LR Test RMSE: 0.2974, LR Test MAE: 0.0603
Division 14 : LR Test RMSE: 0.2633, LR Test MAE: 0.0626
Division 15 : LR Test RMSE: 0.3660, LR Test MAE: 0.0855
Division 16 : LR Test RMSE: 0.1734, LR Test MAE: 0.0409
Division 17 : LR Test RMSE: 0.2092, LR Test MAE: 0.0498
Division 18 : LR Test RMSE: 0.2422, LR Test MAE: 0.0556
D

In [ ]:
lgb_df = pd.DataFrame.from_dict(lgb_results, orient='index')
lgb_df.index.name = "Division"

In [ ]:
lgb_df.head(21)

,Division,test rmse,test mae
Division,,,
1,1,0.671789,0.172391
2,2,0.404372,0.081332
3,3,0.363004,0.085849
4,4,0.183662,0.039952
5,5,0.199924,0.043802
6,6,0.247694,0.070912
7,7,0.235879,0.062131
8,8,0.208260,0.048196
9,9,0.202054,0.046524


In [ ]:
#save to CSV
saving_path4 = os.path.join(path,"LGBM_Regressor.csv")
lgb_df.to_csv(saving_path4, index=False)

**3.4 ANN**